# 04 · Model results: OlivEye vs the official aforo

Reads the output of `scripts/train_models.py` (leave-one-campaign-out, ADR 0003) and plots it.
All logic lives in `oliveye.models`; this notebook only loads and draws.

**How to read it**
- *Real production* is the Junta's final figure for each campaign (ADR 0002).
- *Aforo* is the Junta's official forecast, published in October with field fruit counts.
- *OlivEye (Ridge)* is our forecast with data up to August, made **without** the aforo.
- Every prediction is out of sample: the campaign shown was never used to train the model
  that predicts it (nor was the next campaign, whose lag feature is this campaign's production).

In [ ]:
import json
from pathlib import Path

import pandas as pd

from oliveye.models import plots
from oliveye.models.train import baseline_predictions

RUN = Path("../data/processed/model_run")  # written by scripts/train_models.py
run = json.loads((RUN / "run.json").read_text())
table = pd.read_csv(RUN / "table.csv")
predictions = pd.read_csv(RUN / "predictions.csv")
summary = pd.read_csv(RUN / "summary.csv")
per_campaign = pd.read_csv(RUN / "per_campaign.csv", index_col="campaign")
baselines = baseline_predictions(table)
{k: v for k, v in run.items() if k != "coefficients_by_fold"}

## Summary (same rows for every model)

`p_vs_aforo` is an exact sign-flip test on the per-campaign difference in |log error|: with ~9
campaigns, only a very consistent difference reaches small values.

In [ ]:
summary

## Production per campaign

Each province has its own scale (Jaén produces ~4× Granada). The shaded band is the 80 %
split-conformal interval of the model.

In [ ]:
fig = plots.production_vs_aforo(
    predictions, baselines, interval_factor=run["interval_80pct_conformal_factor"]
)

## Error per campaign

Mean |log error| of both provinces (0.1 ≈ 10 % off). The table below is the same data, for
readers who cannot tell the colours apart.

In [ ]:
fig = plots.error_by_campaign(per_campaign)

In [ ]:
per_campaign.round(3)

## Coefficients across folds

Standardised anomaly coefficients (one per fold). Stable signs across folds mean the effect is
not driven by a single campaign; `coef_level_gap_granada` is the unpenalised log-level gap (Granada − Jaén), not a slope.

In [ ]:
pd.DataFrame(run["coefficients_by_fold"]).round(3)